# Ejercicio 07: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
pip install beir --no-deps

  Using cached beir-2.2.0-py3-none-any.whl.metadata (28 kB)
Using cached beir-2.2.0-py3-none-any.whl (77 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

c:\Users\ladol\anaconda3\Lib\site-packages\beir\util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets\\scifact'

In [4]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [5]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [6]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [7]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [8]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [10]:
from rank_bm25 import BM25Okapi
import numpy as np
import re

def tokenizar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    return texto.split()

# armamos el texto de cada doc juntando titulo y cuerpo
df_corpus["texto_completo"] = df_corpus["title"].fillna("") + " " + df_corpus["text"].fillna("")

corpus_tokens = [tokenizar(t) for t in df_corpus["texto_completo"]]
bm25 = BM25Okapi(corpus_tokens)

# para mapear posicion en la matriz <-> doc_id
doc_ids = df_corpus["doc_id"].tolist()

In [11]:
def buscar_bm25(query, top_k=10):
    q_tokens = tokenizar(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(doc_ids[i], scores[i]) for i in top_idx]

In [12]:
res = buscar_bm25(df_queries.loc[df_queries["query_id"] == "133", "query"].values[0])
for doc_id, score in res:
    print(doc_id, round(score, 3))

5270265 54.351
26688294 53.963
19752008 53.615
45764440 52.882
16280642 52.423
12785130 51.406
5914739 50.5
11200685 49.224
37964706 48.722
35660758 48.453


In [13]:
# relevantes por query: {query_id: set(doc_ids relevantes)}
relevantes = {}
for qid, grupo in df_qrels[df_qrels["relevance"] > 0].groupby("query_id"):
    relevantes[qid] = set(grupo["doc_id"])

In [14]:
def recall_at_k(recuperados, relevantes_q, k=10):
    if not relevantes_q:
        return None
    top = [doc_id for doc_id, _ in recuperados[:k]]
    encontrados = sum(1 for d in top if d in relevantes_q)
    return encontrados / len(relevantes_q)

def ndcg_at_k(recuperados, relevantes_q, k=10):
    if not relevantes_q:
        return None
    top = [doc_id for doc_id, _ in recuperados[:k]]
    dcg = 0.0
    for i, d in enumerate(top):
        if d in relevantes_q:
            dcg += 1 / np.log2(i + 2)
    # idcg: caso ideal, todos los relevantes arriba
    n_rel = min(len(relevantes_q), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(n_rel))
    return dcg / idcg if idcg > 0 else 0.0

In [15]:
recalls = []
ndcgs = []

for _, fila in df_queries.iterrows():
    qid = fila["query_id"]
    if qid not in relevantes:
        continue
    res = buscar_bm25(fila["query"], top_k=10)
    recalls.append(recall_at_k(res, relevantes[qid], k=10))
    ndcgs.append(ndcg_at_k(res, relevantes[qid], k=10))

print(f"Queries evaluadas: {len(recalls)}")
print(f"Recall@10: {np.mean(recalls):.4f}")
print(f"nDCG@10:   {np.mean(ndcgs):.4f}")

Queries evaluadas: 300
Recall@10: 0.7757
nDCG@10:   0.6523


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [16]:
# para cada query guardamos sus top-100 candidatos de bm25 (doc_id y score)
candidatos = {}
for _, fila in df_queries.iterrows():
    qid = fila["query_id"]
    candidatos[qid] = buscar_bm25(fila["query"], top_k=100)

Verifica que un candidato cualquiera tenga sus 100 docs

In [17]:
print(len(candidatos["133"]))
print(candidatos["133"][:3])

100
[('5270265', 54.350569622645835), ('26688294', 53.96264643103443), ('19752008', 53.61472905223674)]


In [18]:
from sentence_transformers import CrossEncoder

modelo_ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\ladol\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ladol\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [19]:
# diccionario doc_id -> texto, para no buscar en el dataframe en cada iteracion
texto_por_doc = dict(zip(df_corpus["doc_id"], df_corpus["texto_completo"]))

def rerank_cross_encoder(query, candidatos_query, top_k=10):
    pares = [(query, texto_por_doc[doc_id]) for doc_id, _ in candidatos_query]
    scores = modelo_ce.predict(pares)
    # ordenamos por score del cross-encoder
    ordenados = sorted(zip(candidatos_query, scores), key=lambda x: x[1], reverse=True)
    return [(doc_id, float(score)) for (doc_id, _), score in ordenados[:top_k]]

Prueba con la query 133 para ver el efecto:

In [20]:
qid = "133"
query_texto = df_queries.loc[df_queries["query_id"] == qid, "query"].values[0]

reranked = rerank_cross_encoder(query_texto, candidatos[qid], top_k=10)

print(f"Relevantes reales: {relevantes[qid]}\n")
print("Top 10 después del cross-encoder:")
for i, (doc_id, score) in enumerate(reranked, 1):
    marca = " <-- RELEVANTE" if doc_id in relevantes[qid] else ""
    print(f"{i:2}. {doc_id}  score={score:.3f}{marca}")

Relevantes reales: {'38485364', '17934082', '6969753', '12640810', '16280642'}

Top 10 después del cross-encoder:
 1. 35660758  score=1.601
 2. 12640810  score=0.360 <-- RELEVANTE
 3. 16280642  score=-0.006 <-- RELEVANTE
 4. 36345185  score=-1.687
 5. 6969753  score=-1.744 <-- RELEVANTE
 6. 9507605  score=-2.607
 7. 86694016  score=-2.698
 8. 14328288  score=-3.105
 9. 21551568  score=-3.233
10. 42708716  score=-3.247


Tabla comparativa

In [21]:
def comparar_top10(qid, ranking_bm25, ranking_reranked):
    top10_bm25 = [doc_id for doc_id, _ in ranking_bm25[:10]]
    top10_rerank = [doc_id for doc_id, _ in ranking_reranked[:10]]

    # posicion en cada ranking (1-indexed), None si no esta
    pos_bm25 = {d: i+1 for i, d in enumerate(top10_bm25)}
    pos_rerank = {d: i+1 for i, d in enumerate(top10_rerank)}

    todos = set(top10_bm25) | set(top10_rerank)
    filas = []
    for doc_id in todos:
        p_bm25 = pos_bm25.get(doc_id)
        p_rerank = pos_rerank.get(doc_id)
        if p_bm25 is None:
            cambio = "entró"
        elif p_rerank is None:
            cambio = "salió"
        elif p_bm25 == p_rerank:
            cambio = "igual"
        else:
            cambio = f"{p_bm25} → {p_rerank}"
        filas.append({
            "doc_id": doc_id,
            "pos_bm25": p_bm25,
            "pos_rerank": p_rerank,
            "cambio": cambio,
            "relevante": doc_id in relevantes[qid]
        })
    return pd.DataFrame(filas).sort_values(by="pos_rerank", na_position="last")

comparar_top10("133", candidatos["133"], reranked)

,doc_id,pos_bm25,pos_rerank,cambio,relevante
15,35660758,10.0,1.0,10 → 1,False
9,12640810,NaN,2.0,entró,True
17,16280642,5.0,3.0,5 → 3,True
4,36345185,NaN,4.0,entró,False
8,6969753,NaN,5.0,entró,True
5,9507605,NaN,6.0,entró,False
12,86694016,NaN,7.0,entró,False
6,14328288,NaN,8.0,entró,False
14,21551568,NaN,9.0,entró,False
3,42708716,NaN,10.0,entró,False


Re-rankea las 300 queries y guarda el resultado (como demora lo ejecutamos primero)

In [23]:
from tqdm import tqdm

rerank_ce = {}
for _, fila in tqdm(df_queries.iterrows(), total=len(df_queries)):
    qid = fila["query_id"]
    rerank_ce[qid] = rerank_cross_encoder(fila["query"], candidatos[qid], top_k=100)

100%|██████████| 300/300 [20:42<00:00,  4.14s/it]


Verificación rápida que tiene 300 entradas

In [24]:
print(len(rerank_ce))
print(len(rerank_ce["133"]))

300
100


In [25]:
import pickle

with open("rerank_ce.pkl", "wb") as f:
    pickle.dump(rerank_ce, f)

## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [27]:
from beir.datasets.data_loader import GenericDataLoader

# el corpus es el mismo, solo nos interesan queries y qrels de train
_, queries_train, qrels_train = GenericDataLoader(dataset_path).load(split="train")

print(f"Queries de entrenamiento: {len(queries_train)}")
print(f"Qrels de entrenamiento: {sum(len(d) for d in qrels_train.values())}")

  0%|          | 0/5183 [00:00<?, ?it/s]

Queries de entrenamiento: 809
Qrels de entrenamiento: 919


In [28]:
# pre-calculamos el idf de cada termino del corpus (lo necesitamos para una feature)
# bm25 ya lo tiene calculado internamente como bm25.idf
idf_corpus = bm25.idf

# guardamos titulo y texto por doc_id para acceso rapido
titulo_por_doc = dict(zip(df_corpus["doc_id"], df_corpus["title"].fillna("")))

def extraer_features(query, doc_id, score_bm25):
    q_tokens = tokenizar(query)
    doc_texto = texto_por_doc[doc_id]
    doc_tokens = tokenizar(doc_texto)
    titulo_tokens = tokenizar(titulo_por_doc[doc_id])

    q_set = set(q_tokens)
    doc_set = set(doc_tokens)
    titulo_set = set(titulo_tokens)

    overlap = len(q_set & doc_set)
    overlap_titulo = len(q_set & titulo_set)
    idf_sum = sum(idf_corpus.get(t, 0) for t in q_set if t in doc_set)

    return [
        score_bm25,
        len(doc_tokens),
        len(q_tokens),
        overlap,
        overlap / len(q_set) if q_set else 0,
        overlap_titulo,
        idf_sum,
    ]

NOMBRES_FEATURES = [
    "bm25", "len_doc", "len_query", "overlap",
    "overlap_norm", "overlap_titulo", "idf_sum"
]

In [29]:
query_133 = df_queries.loc[df_queries["query_id"] == "133", "query"].values[0]
doc_id_test, score_test = candidatos["133"][0]

features_test = extraer_features(query_133, doc_id_test, score_test)
for nombre, valor in zip(NOMBRES_FEATURES, features_test):
    print(f"{nombre:20} {valor:.3f}")

bm25                 54.351
len_doc              165.000
len_query            22.000
overlap              10.000
overlap_norm         0.526
overlap_titulo       3.000
idf_sum              28.809


In [30]:
import pandas as pd

X_train = []
y_train = []
group_train = []

for qid, q_texto in tqdm(queries_train.items(), total=len(queries_train)):
    relevantes_qid = set(qrels_train.get(qid, {}).keys())
    if not relevantes_qid:
        continue

    cands_qid = buscar_bm25(q_texto, top_k=100)

    n_filas = 0
    for doc_id, score_bm25 in cands_qid:
        feats = extraer_features(q_texto, doc_id, score_bm25)
        label = 1 if doc_id in relevantes_qid else 0
        X_train.append(feats)
        y_train.append(label)
        n_filas += 1

    group_train.append(n_filas)

X_train = pd.DataFrame(X_train, columns=NOMBRES_FEATURES)
y_train = pd.Series(y_train)

print(f"Filas: {len(X_train)}")
print(f"Grupos (queries): {len(group_train)}")
print(f"Relevantes encontrados en candidatos: {y_train.sum()}")

100%|██████████| 809/809 [00:56<00:00, 14.43it/s]


Filas: 80900
Grupos (queries): 809
Relevantes encontrados en candidatos: 830


In [31]:
import lightgbm as lgb

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)

ranker.fit(
    X_train,
    y_train,
    group=group_train,
)

LGBMRanker(learning_rate=0.05, metric='ndcg', n_estimators=200,
           objective='lambdarank', random_state=42, verbose=-1)

In [32]:
importancias = pd.DataFrame({
    "feature": NOMBRES_FEATURES,
    "importancia": ranker.feature_importances_
}).sort_values("importancia", ascending=False)

importancias

,feature,importancia
1,len_doc,1476
0,bm25,1315
6,idf_sum,1146
2,len_query,662
4,overlap_norm,657
5,overlap_titulo,374
3,overlap,370


In [33]:
def rerank_ltr(query, candidatos_query, top_k=10):
    feats = [extraer_features(query, doc_id, score) for doc_id, score in candidatos_query]
    X = pd.DataFrame(feats, columns=NOMBRES_FEATURES)
    scores = ranker.predict(X)
    ordenados = sorted(zip(candidatos_query, scores), key=lambda x: x[1], reverse=True)
    return [(doc_id, float(score)) for (doc_id, _), score in ordenados[:top_k]]

Prueba con la query 133, igual que con el cross-encoder:

In [34]:
reranked_ltr = rerank_ltr(query_133, candidatos["133"], top_k=10)

print(f"Relevantes reales: {relevantes['133']}\n")
print("Top 10 después del LTR:")
for i, (doc_id, score) in enumerate(reranked_ltr, 1):
    marca = " <-- RELEVANTE" if doc_id in relevantes["133"] else ""
    print(f"{i:2}. {doc_id}  score={score:.3f}{marca}")

Relevantes reales: {'38485364', '17934082', '6969753', '12640810', '16280642'}

Top 10 después del LTR:
 1. 19752008  score=1.698
 2. 5270265  score=1.598
 3. 26688294  score=1.543
 4. 5914739  score=1.076
 5. 16280642  score=0.686 <-- RELEVANTE
 6. 45764440  score=0.361
 7. 35660758  score=-0.622
 8. 8771704  score=-0.857
 9. 37964706  score=-0.864
10. 12640810  score=-0.954 <-- RELEVANTE


In [35]:
comparar_top10("133", candidatos["133"], reranked_ltr)

,doc_id,pos_bm25,pos_rerank,cambio,relevante
5,19752008,3.0,1.0,3 → 1,False
0,5270265,1.0,2.0,1 → 2,False
2,26688294,2.0,3.0,2 → 3,False
1,5914739,7.0,4.0,7 → 4,False
11,16280642,5.0,5.0,igual,True
7,45764440,4.0,6.0,4 → 6,False
9,35660758,10.0,7.0,10 → 7,False
6,8771704,NaN,8.0,entró,False
8,37964706,9.0,9.0,igual,False
4,12640810,NaN,10.0,entró,True


In [36]:
rerank_ltr_all = {}
for _, fila in tqdm(df_queries.iterrows(), total=len(df_queries)):
    qid = fila["query_id"]
    rerank_ltr_all[qid] = rerank_ltr(fila["query"], candidatos[qid], top_k=100)

print(len(rerank_ltr_all))
print(len(rerank_ltr_all["133"]))

100%|██████████| 300/300 [00:04<00:00, 69.39it/s] 

300
100


Guardar el LTR

In [37]:
with open("rerank_ltr.pkl", "wb") as f:
    pickle.dump(rerank_ltr_all, f)

## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [38]:
def average_precision(recuperados, relevantes_q):
    if not relevantes_q:
        return None
    aciertos = 0
    suma_precisiones = 0.0
    for i, (doc_id, _) in enumerate(recuperados, 1):
        if doc_id in relevantes_q:
            aciertos += 1
            suma_precisiones += aciertos / i
    if aciertos == 0:
        return 0.0
    return suma_precisiones / len(relevantes_q)

In [39]:
def evaluar_sistema(rankings_por_query):
    recalls, ndcgs, aps = [], [], []
    for _, fila in df_queries.iterrows():
        qid = fila["query_id"]
        if qid not in relevantes:
            continue
        ranking = rankings_por_query[qid]
        recalls.append(recall_at_k(ranking, relevantes[qid], k=10))
        ndcgs.append(ndcg_at_k(ranking, relevantes[qid], k=10))
        aps.append(average_precision(ranking, relevantes[qid]))
    return {
        "Recall@10": np.mean(recalls),
        "nDCG@10": np.mean(ndcgs),
        "MAP": np.mean(aps),
    }

# armamos los rankings de bm25 a partir de los candidatos top-100 que ya teniamos
rerank_bm25 = candidatos  # bm25 ya esta ordenado por score

resultados = pd.DataFrame({
    "BM25 (baseline)": evaluar_sistema(rerank_bm25),
    "LTR (LightGBM)": evaluar_sistema(rerank_ltr_all),
    "Cross-encoder": evaluar_sistema(rerank_ce),
}).T

resultados

,Recall@10,nDCG@10,MAP
BM25 (baseline),0.775667,0.652285,0.613060
LTR (LightGBM),0.801333,0.704474,0.670959
Cross-encoder,0.793944,0.676531,0.637451


## Conclusiones

### Sobre el pipeline de dos etapas

La arquitectura implementada sigue el patrón estándar descrito en el material de clase, una primera etapa rápida (BM25) que recupera 100 candidatos por consulta, y una segunda etapa más costosa que reordena ese conjunto. Esta separación tiene una justificación práctica clara — aplicar el cross-encoder o el LTR sobre los 5183 documentos del corpus sería inviable, mientras que sobre 100 candidatos resulta perfectamente manejable.

### Sobre el baseline BM25

BM25 obtuvo nDCG@10 = 0.6523 sobre las 300 queries de scifact, un valor consistente con los reportados en el benchmark oficial de BEIR. El Recall@10 = 0.7757 muestra que el 77% de los documentos relevantes ya están en el top 10 sin necesidad de re-ranking. Esto confirma lo descrito en clases, el recall puede ser alto incluso cuando el orden interno del ranking no es óptimo, y es justamente esa brecha entre recall y nDCG la que motiva re-ranking.

### Sobre el cross-encoder

El cross-encoder `ms-marco-MiniLM-L-6-v2` mejoró el baseline (nDCG@10 = 0.6765, +2.4 puntos) procesando los pares (query, documento) con atención cruzada. El análisis de la query 133 mostró que reordena el ranking agresivamente: rescató 2 documentos relevantes que BM25 tenía fuera del top 10 y movió otro de la posición 5 a la 3, con 8 documentos entrando y 8 saliendo del top 10. Este comportamiento es consistente con la capacidad del modelo de evaluar la relevancia semántica de cada par, más allá de coincidencias léxicas.

### Sobre el LTR

El modelo LightGBM con objetivo `lambdarank` (listwise, alineado con nDCG según las diapositivas) fue entrenado sobre el split train de scifact con 7 features manuales inspiradas en las diapositivas: BM25, longitud del documento, longitud de la query, overlap de términos, overlap normalizado, overlap en el título, y suma de IDF de los términos coincidentes. Las features más informativas según el modelo fueron BM25, IDF-sum y longitud del documento.

El LTR resultó ser **más conservador** que el cross-encoder: en la query 133 solo 2 documentos entraron y 2 salieron del top 10, y los movimientos fueron pequeños (±1 a 3 posiciones). Esto se explica porque el LTR opera sobre features lexicales que son refinamientos de la misma señal que ya capturó BM25, mientras que el cross-encoder evalúa cada par desde cero con un transformer.

### Sobre la comparación final

| Sistema | Recall@10 | nDCG@10 | MAP |
|---|---|---|---|
| BM25 (baseline) | 0.7757 | 0.6523 | 0.6131 |
| LTR (LightGBM) | 0.8013 | 0.7045 | 0.6710 |
| Cross-encoder | 0.7939 | 0.6765 | 0.6375 |

Ambos re-rankers superaron al baseline en las tres métricas: dos sistemas que recuperan el mismo top-100 producen rankings distintos según cómo ordenen ese conjunto, y la métrica nDCG@10 refleja esa diferencia.

El resultado más interesante es que el LTR superó al cross-encoder en este dataset. Las diapositivas sugieren que los modelos neurales suelen ser más precisos que las features manuales, así que este resultado merece análisis. Tres factores lo explican:

1. **Mismatch de dominio.** El cross-encoder fue entrenado en MS-MARCO (queries web generales), mientras que scifact contiene literatura biomédica con vocabulario técnico muy especializado. El material de la clase advierte que cuanto más técnico es el dominio, más importante es adaptar el modelo, y aquí no hubo adaptación.
2. **Ventaja de entrenamiento del LTR.** El LTR fue entrenado con 809 queries del propio scifact (split train), lo que le dio acceso directo a la distribución del dominio. El cross-encoder no vio nada de scifact en su entrenamiento.
3. **Tipo de query.** En el material, el re-ranking neuronal aporta más cuando la consulta es ambigua o larga. Las queries de scifact son enunciados científicos específicos con vocabulario denso, donde las señales léxicas que el LTR sabe explotar (BM25, IDF, overlap) ya son altamente discriminativas.

### Reflexión final

Re-ranking no es una técnica universal que siempre mejore por el solo hecho de ser más sofisticada. El re-ranking corrige el orden localmente dentro del conjunto de candidatos, pero su efectividad depende críticamente de la etapa 1 y del alineamiento entre el modelo de re-ranking y el dominio de la tarea. En este experimento, un LTR con features simples entrenado en el dominio correcto resultó más efectivo que un cross-encoder neural entrenado en un dominio distinto, lo que constituye una evidencia concreta de que la elección del re-ranker debe responder a las características de la tarea y no a la complejidad nominal del modelo.